# PRE-DATA PROCESSING OF DRUG DATASETS

In [1]:
import pandas
import os

In [2]:
# DIRECTORIES
notebook_dir = os.getcwd()
base_dir = os.path.dirname(notebook_dir) # the notebook is in a subdirectory of the base directory
input_dir = os.path.join(base_dir, 'data', 'input')
print(f"Base directory: {base_dir}")
print(f"Input directory: {input_dir}")

Base directory: c:\Users\viviamsb\OneDrive - NTNU\PhD Folder\Pipeline\DREXPA
Input directory: c:\Users\viviamsb\OneDrive - NTNU\PhD Folder\Pipeline\DREXPA\data\input


In [3]:
# FILES
drugdata_file = input_dir + '/jaaks/jaaks_original_screen_all_tissues_fitted_data.csv'
drugdata_df = pandas.read_csv(drugdata_file, sep=',')
drugdata_df.columns

C:\Users\viviamsb\AppData\Local\Temp\ipykernel_28668\1077708944.py:3: DtypeWarning: Columns (6,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  drugdata_df = pandas.read_csv(drugdata_file, sep=',')


Index(['BARCODE', 'COMBI_ID', 'Tissue', 'CELL_LINE_NAME', 'SIDM', 'COSMIC_ID',
       'ANCHOR_ID', 'ANCHOR_NAME', 'ANCHOR_TARGET', 'ANCHOR_PATHWAY',
       'ANCHOR_DRUG_TYPE', 'ANCHOR_Clin_Rel', 'ANCHOR_CONC', 'LIBRARY_ID',
       'LIBRARY_NAME', 'LIBRARY_TARGET', 'LIBRARY_PATHWAY',
       'LIBRARY_DRUG_TYPE', 'LIBRARY_Clin_Rel', 'LIBRARY_CONC',
       'ANCHOR_VIABILITY', 'LIBRARY_RMSE', 'LIBRARY_EMAX', 'LIBRARY_XMID',
       'LIBRARY_XMID_uM', 'LIBRARY_AUC', 'LIBRARY_fAUC', 'SYNERGY_RMSE',
       'SYNERGY_EXP_EMAX', 'SYNERGY_OBS_EMAX', 'SYNERGY_DELTA_EMAX',
       'SYNERGY_XMID', 'SYNERGY_XMID_uM', 'SYNERGY_DELTA_XMID',
       'SYNERGY_DELTA_XMID_uM', 'SYNERGY_EXP_AUC', 'SYNERGY_OBS_AUC',
       'SYNERGY_EXP_fAUC', 'SYNERGY_OBS_fAUC', 'SYNERGY_DELTA_AUC',
       'SYNERGY_DELTA_fAUC', 'Synergy', 'DAY1_NORM_MEAN', 'DAY1_NORM_SD',
       'GROWTH_RATE', 'DOUBLING_TIME'],
      dtype='object')

In [4]:
drugdata_df.CELL_LINE_NAME.nunique()

125

In [4]:
# CLEAN CELL LINE NAMES
print(f'Total unique cell lines in dataset before cleaning: {drugdata_df["CELL_LINE_NAME"].nunique()}')
drugdata_df['CELL_LINE_NAME'] = drugdata_df['CELL_LINE_NAME'].str.upper()
# Remove '-' or '+' from cell line names
drugdata_df['CELL_LINE_NAME'] = drugdata_df['CELL_LINE_NAME'].str.replace('-', '', regex=False)
drugdata_df['CELL_LINE_NAME'] = drugdata_df['CELL_LINE_NAME'].str.replace('+', '', regex=False)
print(f'Total unique cell lines in dataset after cleaning: {drugdata_df["CELL_LINE_NAME"].nunique()}')
# UNIQUE CELL LINES IN DATASET
clines = drugdata_df['CELL_LINE_NAME'].unique().tolist()
clines.sort()

Total unique cell lines in dataset before cleaning: 125
Total unique cell lines in dataset after cleaning: 125


In [5]:
# UNIQUE DRUGS IN DATASET
drug_names = drugdata_df[['ANCHOR_NAME', 'LIBRARY_NAME']].copy()
drug_names['ANCHOR_NAME'] = drug_names['ANCHOR_NAME'].str.upper()
drug_names['LIBRARY_NAME'] = drug_names['LIBRARY_NAME'].str.upper()

# SYNONYMS
updatedNames_dict = {
    'Nutlin-3a (-)': 'NUTLIN-3',
    'PD173074': 'PD-173074',
    'GSK269962A': 'GSK-269962A',
    'SB216763': 'SB-216763',
    'NU7441': 'NU-7441',
    'SB505124': 'SB-505124',
    'ZM447439': 'ZM-447439'
}
drug_names['ANCHOR_NAME'] = drug_names['ANCHOR_NAME'].replace(updatedNames_dict)
drug_names['LIBRARY_NAME'] = drug_names['LIBRARY_NAME'].replace(updatedNames_dict)

# Omit 'Afatinib|Trametinib' drug from the DataFrame
omitted_drug = 'Afatinib|Trametinib'
drug_names = drug_names[~drug_names['ANCHOR_NAME'].str.contains(omitted_drug)]
drug_names = drug_names[~drug_names['LIBRARY_NAME'].str.contains(omitted_drug)]

# Make list of total drugs in dataset
drug_list = sorted(set(drug_names['ANCHOR_NAME'].tolist() + drug_names['LIBRARY_NAME'].tolist()))
print(f'Total unique drugs in dataset: {len(drug_list)}')

# Save list as dataframe of single column 'drug_name'
drug_df = pandas.DataFrame(drug_list, columns=['drug_name'])
drugnames_file = input_dir + '/jaaks/drug_names.txt'
drug_df.to_csv(drugnames_file, index=False, header=True)

Total unique drugs in dataset: 65


In [17]:
# DOSES
# Change concentration units from uM to nM
drugdata_df['ANCHOR_CONC'] = pandas.to_numeric(drugdata_df['ANCHOR_CONC'], errors='coerce') * 1000
drugdata_df['LIBRARY_CONC'] = pandas.to_numeric(drugdata_df['LIBRARY_CONC'], errors='coerce') * 1000

In [8]:
# UNIQUE TRIPLETS IN DATASET
triplets = drugdata_df[['CELL_LINE_NAME', 'ANCHOR_NAME', 'LIBRARY_NAME']].copy()
triplets = triplets.drop_duplicates()
triplets = triplets.reset_index(drop=True)
print(f'Total unique (cell_line, drugA, drugB) triplets in dataset: {len(triplets)}')

Total unique (cell_line, drugA, drugB) triplets in dataset: 108259


In [10]:
# TISSUE INFORMATION
# Extract tissue information from cell line names
tissue_info = drugdata_df[['Tissue', 'CELL_LINE_NAME']].copy()
tissue_info = tissue_info.drop_duplicates()
tissue_info = tissue_info.reset_index(drop=True)
tissue_info.to_csv(input_dir + '/jaaks/tissue_cline.csv', index=False, header=True)

In [18]:
# SYNERGY DATA
synergies = drugdata_df[['Tissue', 'CELL_LINE_NAME', 'ANCHOR_NAME', 'LIBRARY_NAME', 'ANCHOR_CONC', 'LIBRARY_CONC', 'Synergy']].copy()
rename_dict = {
    'CELL_LINE_NAME': 'cell_line',
    'ANCHOR_NAME': 'drug_name_A',
    'LIBRARY_NAME': 'drug_name_B',
    'ANCHOR_CONC': 'conc_A',
    'LIBRARY_CONC': 'conc_B',
    'Synergy': 'synergy'
}
synergies = synergies.rename(columns=rename_dict)
synergies.to_csv(input_dir + '/jaaks/synergy_data.csv', index=False, header=True)